# Crop Recommendation System

This notebook trains and tests a separate crop recommendation model from `../../recommend_data/Crop_recommendation.csv`. It recommends a suitable crop from soil nutrients and weather values.

## 1. Imports

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("ipywidgets is unavailable. Use the manual input cell at the bottom.")

## 2. Load Dataset

In [2]:
DATA_PATH = Path("../../recommend_data/Crop_recommendation.csv")
MODEL_PATH = Path("../../models/crop_recommendation_model.pkl")
METRICS_PATH = Path("../../models/model_metrics.json")

df = pd.read_csv(DATA_PATH).dropna()
print(df.shape)
df.head()

(2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


## 3. Train Recommendation Model

In [3]:
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
target = "label"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])
model.fit(X_train, y_train)
print("Crop recommendation model training complete.")

Crop recommendation model training complete.


## 4. Evaluate and Save

In [4]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
metrics = {
    "rows": int(len(df)),
    "features": features,
    "target": target,
    "test_size": int(len(X_test)),
    "accuracy": float(accuracy),
    "labels": sorted(df[target].unique().tolist()),
}
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, predictions))

MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(model, MODEL_PATH, compress=3)

all_metrics = {}
if METRICS_PATH.exists():
    all_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
all_metrics["crop_recommendation_model"] = metrics
METRICS_PATH.write_text(json.dumps(all_metrics, indent=2), encoding="utf-8")
print(f"Saved recommendation model to: {MODEL_PATH}")

Accuracy: 0.9955
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      papa

## 5. Recommend Crop With Form

In [5]:
def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    row = pd.DataFrame([{
        "N": float(N),
        "P": float(P),
        "K": float(K),
        "temperature": float(temperature),
        "humidity": float(humidity),
        "ph": float(ph),
        "rainfall": float(rainfall),
    }])
    prediction = model.predict(row)[0]
    probabilities = model.predict_proba(row)[0]
    classes = model.named_steps["model"].classes_
    top = pd.DataFrame({"crop": classes, "confidence": probabilities}).sort_values("confidence", ascending=False).head(5)
    return row, prediction, top

if widgets is None:
    print("ipywidgets unavailable. Use the manual input cell below.")
else:
    controls = {
        "N": widgets.FloatText(value=90, description="N"),
        "P": widgets.FloatText(value=42, description="P"),
        "K": widgets.FloatText(value=43, description="K"),
        "temperature": widgets.FloatText(value=21.0, description="Temp"),
        "humidity": widgets.FloatText(value=82.0, description="Humidity"),
        "ph": widgets.FloatText(value=6.5, description="pH"),
        "rainfall": widgets.FloatText(value=203.0, description="Rainfall"),
    }
    button = widgets.Button(description="Recommend Crop", button_style="success")
    output = widgets.Output()

    def on_click(_):
        with output:
            output.clear_output()
            row, prediction, top = recommend_crop(**{k: v.value for k, v in controls.items()})
            display(row)
            print(f"Recommended crop: {prediction}")
            display(top.style.format({"confidence": "{:.2%}"}).hide(axis="index"))

    button.on_click(on_click)
    display(widgets.VBox(list(controls.values()) + [button, output]))

## Optional: Manual Input Fallback

In [6]:
manual_input = {
    "N": 90,
    "P": 42,
    "K": 43,
    "temperature": 20.88,
    "humidity": 82.00,
    "ph": 6.50,
    "rainfall": 202.94,
}
row, prediction, top = recommend_crop(**manual_input)
display(row)
print(f"Recommended crop: {prediction}")
display(top.style.format({"confidence": "{:.2%}"}).hide(axis="index"))

,N,P,K,temperature,humidity,ph,rainfall
0,90.0,42.0,43.0,20.88,82.0,6.5,202.94


Recommended crop: rice


crop,confidence
rice,93.50%
jute,6.50%
blackgram,0.00%
chickpea,0.00%
apple,0.00%
